In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from dataclasses import dataclass
from lunanav.constants import GM_MOON, R_MOON, RAD_TO_DEG
from lunanav.sim.simulator import SimParams, RigidBody, run_sim, SimResults, reverse_sim_results
from lunanav.sim.sensors import (
    SensorEnvironment, Sensor, SensorSuite, SensorName,
    accelerometer_sensor, gyroscope_sensor,
    laser_altimeter_sensor, laser_velocity_sensor,
    star_tracker_sensor, doppler_sensor, sat_range_tracker_sensor
)
from lunanav.estimation.ekf import ekf_predict, ekf_update, Qd_from_accel_white
from lunanav.sim.quaternion import unitize_state, angle_axis_to_q, quat_apply, conj
from lunanav.plotting import plot_control_effort, plot_state_vector_combined, plot_state_vector_combined_log, plot_state_vector
from lunanav.visualization import visualize_trajectory, plot_measurements, plot_attitude_relative_vertical, plot_filter_confidence, obsv_verbose
from lunanav.sim.sensors import get_los_vectors

In [ ]:
from lunanav.loaders import load_trajectory, load_sim_result, load_ekf_result
from lunanav.sim.generate import SatPosVel, make_sat_arrs

TRAJ_FILE = "data/trajectories/ilqr_easier.json"
traj = load_trajectory(TRAJ_FILE)

sim_result = load_sim_result("data/simresults/tilty.json")
ekf_result = load_ekf_result("data/ekfresults/ekf_result.json")

# Unpack for convenience
dt = traj.dt
t_max = traj.T
mass_kg = traj.mass_kg
I = traj.I.reshape((3, 3))

s_true = sim_result.s_true
t = sim_result.t_arr
n = sim_result.nsteps

# Rebuild results object for compatibility
from lunanav.sim.simulator import SimResults
results = SimResults(n)
results.t        = sim_result.t_arr
results.states   = sim_result.s_true
results.force_N  = sim_result.measurements["accelerometer"]["truth"] * mass_kg
results.nsteps   = n

# Load measurements
from lunanav.sim.sensors import SensorName
measurements_noisy = {
    SensorName(k): np.array(v["noisy"]) for k, v in sim_result.measurements.items()
}
measurements_clean = {
    SensorName(k): np.array(v["truth"]) for k, v in sim_result.measurements.items()
}
print(f"Loaded sim result: {n} steps, dt={dt}s")

In [ ]:
from lunanav.sim.generate import generate_env

# Noise parameters (match main notebook)
sigma_accel = 1
sigma_gyro = 1e-3
sigma_los = 1
sigma_los_vel = 1
sigma_star = 1e-3
sigma_doppler = 0.1
sigma_sat_range_tracker = 10

lander = RigidBody(mass_kg=mass_kg, I=I)
state0 = s_true[0]
sim = SimParams(state0, lander, dt, t_max)

sensor_suite = SensorSuite(sensors={
    SensorName.ACCELEROMETER: accelerometer_sensor(sigma_accel),
    SensorName.GYROSCOPE: gyroscope_sensor(sigma_gyro),
    SensorName.LASER_ALTIMETER: laser_altimeter_sensor(sigma_los),
    SensorName.LASER_VELOCITY: laser_velocity_sensor(sigma_los_vel),
    SensorName.STAR_TRACKER: star_tracker_sensor(sigma_star),
    SensorName.DOPPLER: doppler_sensor(3, sigma_doppler),
    SensorName.RANGE_TRACKER: sat_range_tracker_sensor(3, sigma_sat_range_tracker),
})

doppler_sats = [
    make_sat_arrs(t, altitude=100e3, raan=0, aop=90, inc=90),
    make_sat_arrs(t, altitude=100e3, raan=90, aop=94, inc=94),
    make_sat_arrs(t, altitude=100e3, raan=160, aop=70, inc=86),
]

env_arr = generate_env(results, sim, doppler_sats)
print("Sensor suite and environment ready")

In [ ]:
from lunanav.estimation.ekf import ekf_predict, update_sensor, Qd_from_accel_white
from lunanav.sim.quaternion import unitize_state

# Q matrix
Q6 = Qd_from_accel_white(dt, sigma_accel)
Q_att = np.eye(4) * (sigma_gyro * dt)**2
Q_ang = np.eye(3) * (sigma_gyro * dt)**2
Q_ekf = np.block([
    [Q6,                   np.zeros((6, 4)), np.zeros((6, 3))],
    [np.zeros((4, 6)), Q_att,              np.zeros((4, 3))],
    [np.zeros((3, 6)), np.zeros((3, 4)), Q_ang],
])
print("EKF Q matrix ready")

# Sensor Ablation Study

Run EKF with different sensor subsets to evaluate observability.

In [ ]:
# Define sensor dropout configurations with meaningful combinations
sensor_configs = [
    {
        "name": "All Sensors",
        "description": "All sensors active (baseline)",
        "frequencies": {
            SensorName.LASER_ALTIMETER: 1,
            SensorName.LASER_VELOCITY: 1,
            SensorName.STAR_TRACKER: 1,
            SensorName.DOPPLER: 1,
            SensorName.RANGE_TRACKER: 1,
        }
    },
    {
        "name": "No Altimeter",
        "description": "Height-blind configuration (no laser altitude)",
        "frequencies": {
            SensorName.LASER_ALTIMETER: None,
            SensorName.LASER_VELOCITY: 1,
            SensorName.STAR_TRACKER: 1,
            SensorName.DOPPLER: 1,
            SensorName.RANGE_TRACKER: 1,
        }
    },
    {
        "name": "No Laser Velocity",
        "description": "Without local velocity measurement",
        "frequencies": {
            SensorName.LASER_ALTIMETER: 1,
            SensorName.LASER_VELOCITY: None,
            SensorName.STAR_TRACKER: 1,
            SensorName.DOPPLER: 1,
            SensorName.RANGE_TRACKER: 1,
        }
    },
    {
        "name": "No Star Tracker",
        "description": "No absolute attitude reference",
        "frequencies": {
            SensorName.LASER_ALTIMETER: 1,
            SensorName.LASER_VELOCITY: 1,
            SensorName.STAR_TRACKER: None,
            SensorName.DOPPLER: 1,
            SensorName.RANGE_TRACKER: 1,
        }
    },
    {
        "name": "Laser Only",
        "description": "Terrain-relative navigation (ideal TRN)",
        "frequencies": {
            SensorName.LASER_ALTIMETER: 1,
            SensorName.LASER_VELOCITY: 1,
            SensorName.STAR_TRACKER: None,
            SensorName.DOPPLER: None,
            SensorName.RANGE_TRACKER: None,
        }
    },
    {
        "name": "Laser + Star Tracker",
        "description": "TRN with attitude reference",
        "frequencies": {
            SensorName.LASER_ALTIMETER: 1,
            SensorName.LASER_VELOCITY: 1,
            SensorName.STAR_TRACKER: 1,
            SensorName.DOPPLER: None,
            SensorName.RANGE_TRACKER: None,
        }
    },
    {
        "name": "Vision + Navigation",
        "description": "Star tracker + Doppler + Range (no local sensors)",
        "frequencies": {
            SensorName.LASER_ALTIMETER: None,
            SensorName.LASER_VELOCITY: None,
            SensorName.STAR_TRACKER: 1,
            SensorName.DOPPLER: 1,
            SensorName.RANGE_TRACKER: 1,
        }
    },
    {
        "name": "IMU Only",
        "description": "Dead reckoning (accelerometer + gyro only)",
        "frequencies": {
            SensorName.LASER_ALTIMETER: None,
            SensorName.LASER_VELOCITY: None,
            SensorName.STAR_TRACKER: None,
            SensorName.DOPPLER: None,
            SensorName.RANGE_TRACKER: None,
        }
    },
]

In [ ]:
# Run EKF for each configuration (no initial offset)
no_offset_results_list = []

for config in sensor_configs:
    print(f"\n{'='*60}")
    print(f"Running: {config['name']}")
    print(f"Description: {config['description']}")
    print(f"{'='*60}")
    
    mu_arr = np.zeros((n, 13))
    Sigma_arr = np.zeros((n, 13, 13))
    
    mu_arr[0] = s_true[0]
    Sigma_arr[0] = np.eye(13)
    
    for i in tqdm(range(n - 1), desc=config['name'], leave=False):
        force_B = sim_result.measurements[SensorName.ACCELEROMETER.value]['truth'][i] * mass_kg
        accel_meas = force_B / mass_kg
        gyro_meas = s_true[i, 10:13]
        
        mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
        mu_pred = unitize_state(mu_pred)
        
        env = env_arr[i]
        
        for sensor, freq in config['frequencies'].items():
            mu_pred, Sigma_pred = update_sensor(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements_noisy, i)
        
        mu_arr[i + 1] = mu_pred
        Sigma_arr[i + 1] = Sigma_pred
    
    pos_error = np.linalg.norm(mu_arr[:, 0:3] - s_true[:, 0:3], axis=1)
    vel_error = np.linalg.norm(mu_arr[:, 3:6] - s_true[:, 3:6], axis=1)
    att_error = np.linalg.norm(mu_arr[:, 6:10] - s_true[:, 6:10], axis=1)
    
    no_offset_results_list.append({
        "name": config['name'],
        "description": config['description'],
        "config": config['frequencies'],
        "mu_arr": mu_arr,
        "Sigma_arr": Sigma_arr,
        "pos_error": pos_error,
        "vel_error": vel_error,
        "att_error": att_error,
    })

print("\n" + "="*60)
print("All configurations completed!")
print("="*60)

In [ ]:
from lunanav.sim.quaternion import angle_axis_to_q

# Run EKF for each configuration (with initial offset)
offset_results_list = []

for config in sensor_configs:
    print(f"\n{'='*60}")
    print(f"Running: {config['name']}")
    print(f"Description: {config['description']}")
    print(f"{'='*60}")
    
    mu_arr = np.zeros((n, 13))
    Sigma_arr = np.zeros((n, 13, 13))
    
    mu_arr[0] = s_true[0] + np.array([1e3, -1e3, 1e3, 1e2, 1e2, 1e2, *angle_axis_to_q(90, [0,1,0], True), 10 * RAD_TO_DEG, -14 * RAD_TO_DEG, 30 * RAD_TO_DEG])
    Sigma_arr[0] = np.eye(13)
    
    for i in tqdm(range(n - 1), desc=config['name'], leave=False):
        accel_meas = sim_result.measurements[SensorName.ACCELEROMETER.value]['truth'][i]
        gyro_meas = s_true[i, 10:13]
        
        mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
        mu_pred = unitize_state(mu_pred)
        
        env = env_arr[i]
        
        for sensor, freq in config['frequencies'].items():
            mu_pred, Sigma_pred = update_sensor(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements_noisy, i)
        
        mu_arr[i + 1] = mu_pred
        Sigma_arr[i + 1] = Sigma_pred
    
    pos_error = np.linalg.norm(mu_arr[:, 0:3] - s_true[:, 0:3], axis=1)
    vel_error = np.linalg.norm(mu_arr[:, 3:6] - s_true[:, 3:6], axis=1)
    att_error = np.linalg.norm(mu_arr[:, 6:10] - s_true[:, 6:10], axis=1)
    
    offset_results_list.append({
        "name": config['name'],
        "description": config['description'],
        "config": config['frequencies'],
        "mu_arr": mu_arr,
        "Sigma_arr": Sigma_arr,
        "pos_error": pos_error,
        "vel_error": vel_error,
        "att_error": att_error,
    })

print("\n" + "="*60)
print("All configurations completed!")
print("="*60)

In [ ]:
def plot_sensor_config_comparison(results_list, t):
    """Plot position, velocity, and attitude errors for all sensor configurations."""
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('EKF Error Comparison Across Sensor Configurations', fontsize=14, fontweight='bold')
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(results_list)))
    
    for i, result in enumerate(results_list):
        axs[0].semilogy(t, result['pos_error'], label=result['name'], color=colors[i], linewidth=2)
    axs[0].set_xlabel('Time (s)', fontsize=11)
    axs[0].set_ylabel('Position Error (m)', fontsize=11)
    axs[0].set_title('Position Error', fontsize=12, fontweight='bold')
    axs[0].grid(True, alpha=0.3, which='both')
    axs[0].legend(fontsize=9, loc='best')
    
    for i, result in enumerate(results_list):
        axs[1].semilogy(t, result['vel_error'], label=result['name'], color=colors[i], linewidth=2)
    axs[1].set_xlabel('Time (s)', fontsize=11)
    axs[1].set_ylabel('Velocity Error (m/s)', fontsize=11)
    axs[1].set_title('Velocity Error', fontsize=12, fontweight='bold')
    axs[1].grid(True, alpha=0.3, which='both')
    axs[1].legend(fontsize=9, loc='best')
    
    for i, result in enumerate(results_list):
        axs[2].semilogy(t, result['att_error'], label=result['name'], color=colors[i], linewidth=2)
    axs[2].set_xlabel('Time (s)', fontsize=11)
    axs[2].set_ylabel('Attitude Error', fontsize=11)
    axs[2].set_title('Attitude Error', fontsize=12, fontweight='bold')
    axs[2].grid(True, alpha=0.3, which='both')
    axs[2].legend(fontsize=9, loc='best')
    
    plt.tight_layout()
    plt.show(block=False)
    
    print("\n" + "="*80)
    print("SENSOR CONFIGURATION COMPARISON - FINAL ERROR METRICS")
    print("="*80)
    for result in results_list:
        pos_final = result['pos_error'][-1]
        vel_final = result['vel_error'][-1]
        att_final = result['att_error'][-1]
        pos_mean = np.mean(result['pos_error'][-100:])
        vel_mean = np.mean(result['vel_error'][-100:])
        print(f"\n{result['name']:25s} | {result['description']}")
        print(f"  Position - Final: {pos_final:8.2f} m  | Mean(last): {pos_mean:8.2f} m")
        print(f"  Velocity - Final: {vel_final:8.4f} m/s | Mean(last): {vel_mean:8.4f} m/s")
        print(f"  Attitude - Final: {att_final:8.4f}")

In [ ]:
plot_sensor_config_comparison(offset_results_list, t)

In [ ]:
plot_sensor_config_comparison(no_offset_results_list, t)